# 04 · Download and preprocess AERONET Lunar AOD Level 2.0 (Version 3), all points

**What this notebook does** — the three steps of notebook 01, for the night-time (lunar) product:

1. **Download** the NASA AERONET bulk archive (`.tar.gz`) into `DATA/RAW/` if it is not already there, extract it, and delete the archive.
2. **Read** every station file (`*.lev20`, one per AERONET site) and tidy it.
3. **Export** one agent-ready Parquet table to `DATA/PROCESSED/`.

**Dataset**: AERONET **Lunar AOD Level 2.0** (cloud-screened, quality-assured, pre- and post-field calibration), **Version 3**, **All points**, all sites —
`https://aeronet.gsfc.nasa.gov/data_push/V3/AOD_LUNAR/AOD_Lunar_Level20_All_Points_V3.tar.gz` (~1.0 GB archive → ~8.7 GB extracted, ~523 sites, from 2014).

**All points, not daily averages**: one row = one lunar measurement (a triplet, about every 3 minutes while the Moon is up). The Moon is a faint,
variable source: AOD is retrieved at 440, 500, 675, 870, 1020 and 1640 nm only (no UV), only while the lunar phase angle is within ±90° (the bright
half of each lunar cycle, roughly 12–14 nights a month) and with the Sun more than 8° below the horizon. Level 2.0 lunar AOD was released by AERONET
in June 2025 (Schafer et al. 2026); it uses an empirically corrected ROLO lunar-irradiance model and the solar Version 3 cloud screening.
Besides the AOD columns the file carries the triplet variability per wavelength, Ångström exponents, precipitable water, lunar zenith angle,
optical air mass, lunar phase angle, sensor temperature and the ozone / NO₂ columns used in the processing.

**Too big for one DataFrame** (≈ 8 million rows): the parquet is written **station by station** (a `pyarrow.parquet.ParquetWriter`); a first pass
over the files finds the columns that hold data anywhere, so that the written table has one fixed schema. Everything else is as in notebook 01.


In [1]:
from pathlib import Path
import re
import tarfile
import time

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm
import pyarrow as pa
import pyarrow.parquet as pq

/home/ubuntu/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ARCHIVE = "AOD_Lunar_Level20_All_Points_V3.tar.gz"
URL = "https://aeronet.gsfc.nasa.gov/data_push/V3/AOD_LUNAR/" + ARCHIVE

In [3]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "process_data":          # this notebook lives in notebooks/process_data/
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

In [4]:
RAW_DIR       = PROJECT_ROOT / "DATA" / "RAW"
PROCESSED_DIR = PROJECT_ROOT / "DATA" / "PROCESSED"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
TARBALL     = RAW_DIR / ARCHIVE                                  # deleted after extraction
EXTRACT_DIR = RAW_DIR / ARCHIVE.replace(".tar.gz", "")           # station files land here
OUT_PARQUET = PROCESSED_DIR / "AERONET_Lunar_AOD_L2_AllPoints_V3.parquet"

In [6]:
print(f"URL          : {URL}")
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"EXTRACT_DIR  : {EXTRACT_DIR}")
print(f"OUT_PARQUET  : {OUT_PARQUET}")

URL          : https://aeronet.gsfc.nasa.gov/data_push/V3/AOD_LUNAR/AOD_Lunar_Level20_All_Points_V3.tar.gz
PROJECT_ROOT : /home/ubuntu/ai_for_aeronet
EXTRACT_DIR  : /home/ubuntu/ai_for_aeronet/DATA/RAW/AOD_Lunar_Level20_All_Points_V3
OUT_PARQUET  : /home/ubuntu/ai_for_aeronet/DATA/PROCESSED/AERONET_Lunar_AOD_L2_AllPoints_V3.parquet


## 1 · Download (if needed) → extract → delete the archive


In [7]:
def download(url: str, dest: Path, chunk_size: int = 1 << 20) -> None:
    tmp = dest.with_name(dest.name + ".part")
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get("Content-Length", 0)) or None
        with open(tmp, "wb") as fh, tqdm(total=total, unit="B", unit_scale=True, desc=dest.name) as bar:
            for part in r.iter_content(chunk_size):
                fh.write(part)
                bar.update(len(part))
    tmp.replace(dest)


def extract_tar_gz(archive: Path, dest: Path) -> None:
    dest.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive, "r:gz") as tf:
        try:
            tf.extractall(dest, filter="data")      
        except TypeError:                    
            tf.extractall(dest)


station_files = sorted(EXTRACT_DIR.rglob("*.lev20")) if EXTRACT_DIR.exists() else []

if station_files:
    print(f"Already extracted: {len(station_files)} station files in {EXTRACT_DIR}  →  skipping download.")
else:
    if TARBALL.exists():
        print(f"Found {TARBALL.name} ({TARBALL.stat().st_size / 1e6:,.0f} MB)  →  skipping download.")
    else:
        print(f"Downloading {URL}")
        t0 = time.time()
        download(URL, TARBALL)
        print(f"  done in {time.time() - t0:,.0f} s  ({TARBALL.stat().st_size / 1e6:,.0f} MB)")

    print(f"Extracting → {EXTRACT_DIR}")
    extract_tar_gz(TARBALL, EXTRACT_DIR)
    TARBALL.unlink()
    print(f"Deleted {TARBALL.name}")
    station_files = sorted(EXTRACT_DIR.rglob("*.lev20"))

print(f"{len(station_files):,} station files ready")

Already extracted: 523 station files in /home/ubuntu/ai_for_aeronet/DATA/RAW/AOD_Lunar_Level20_All_Points_V3  →  skipping download.
523 station files ready


## 2 · Read one station file

Each `*.lev20` file is one AERONET site, laid out like the solar AOD files of notebook 01:

```
line 1  AERONET Version 3
line 2  <site name>
line 3  Version 3: Lunar AOD Level 2.0
line 4  (description)
line 5  Contact: PI=<name>; PI Email=<email>
line 6  All Points,UNITS can be found at,,, https://aeronet.gsfc.nasa.gov/new_web/units.html
line 7  <CSV header>   116 columns: Date, Time, Day_of_Year, Day_of_Year(Fraction), AOD_<λ>nm, Precipitable_Water(cm), Triplet_Variability_<λ>, Ångström exponents,
                       Data_Quality_Level, AERONET_Instrument_Number, AERONET_Site_Name, Site_Latitude/Longitude/Elevation, Measurement_Type(solar or lunar),
                       Solar_Zenith_Angle, Lunar_Zenith_Angle, Optical_Air_Mass, Lunar_Phase_Angle, Sensor_Temperature, Ozone, NO2, Last_Date_Processed,
                       Number_of_Wavelengths, Exact_Wavelengths_of_AOD(um)_<λ>nm
line 8+ <CSV rows>, one per measurement, missing values are -999
```

`read_station` is the one of notebook 01.


In [8]:
MISSING = -999.0

def read_station(path: Path) -> pd.DataFrame:
    # Parse one AERONET .lev20 file -> DataFrame (raw AERONET column names, -999 -> NaN,
    # plus `site`, `PI`, `PI_Email` taken from the file header).
    with open(path, encoding="latin-1") as fh:
        header = [next(fh).rstrip("\n") for _ in range(6)]
    site = header[1].strip()
    m = re.search(r"PI=(.*?);\s*PI Email=(.*)", header[4])
    pi, pi_email = (m.group(1).strip(), m.group(2).strip()) if m else (None, None)

    df = pd.read_csv(path, skiprows=6, encoding="latin-1", low_memory=False)
    df = df.mask(df == MISSING)        # -999 → NaN in one step (strings never equal -999; count columns never contain it, so they stay integer)
    df.insert(0, "site", site)
    df["PI"] = pi
    df["PI_Email"] = pi_email
    return df

## 3 · First pass: which columns hold data at all

Many wavelengths are never measured at night (the UV channels, the rarer filters) and the `AOD_Empty` placeholders are always empty.
Counting non-missing values per column over all stations tells which columns to keep, so that every station is written with the same columns.


In [9]:
counts = None
t0 = time.time()
for p in tqdm(station_files, desc="Counting"):
    n = read_station(p).notna().sum()
    counts = n if counts is None else counts.add(n, fill_value=0)
counts = counts.astype(int)
n_rows = int(counts["site"])                                        # `site` is never missing: its count is the number of rows

empty = [c for c in counts.index if counts[c] == 0 or "Empty" in c]
print(f"{n_rows:,} rows in {len(station_files)} station files, counted in {time.time() - t0:,.0f} s")
print(f"dropping {len(empty)} columns that are placeholders or empty at every site:")
print(empty)

Counting:   0%|          | 0/523 [00:00<?, ?it/s]

Counting:   0%|          | 1/523 [00:01<15:41,  1.80s/it]

Counting:   1%|          | 4/523 [00:01<03:12,  2.70it/s]

Counting:   1%|▏         | 7/523 [00:02<01:39,  5.21it/s]

Counting:   2%|▏         | 10/523 [00:02<01:07,  7.59it/s]

...

Counting:  99%|█████████▉| 520/523 [05:32<00:03,  1.15s/it]

Counting: 100%|█████████▉| 522/523 [05:33<00:00,  1.25it/s]

Counting: 100%|██████████| 523/523 [05:33<00:00,  1.29it/s]

Counting: 100%|██████████| 523/523 [05:33<00:00,  1.57it/s]

8,303,752 rows in 523 station files, counted in 334 s
dropping 42 columns that are placeholders or empty at every site:
['AOD_620nm', 'AOD_555nm', 'AOD_551nm', 'AOD_532nm', 'AOD_531nm', 'AOD_400nm', 'AOD_380nm', 'AOD_340nm', 'AOD_Empty', 'AOD_Empty.1', 'AOD_Empty.2', 'AOD_Empty.3', 'AOD_Empty.4', 'Triplet_Variability_620', 'Triplet_Variability_555', 'Triplet_Variability_551', 'Triplet_Variability_532', 'Triplet_Variability_531', 'Triplet_Variability_400', 'Triplet_Variability_380', 'Triplet_Variability_340', 'Triplet_Variability_AOD_Empty', 'Triplet_Variability_AOD_Empty.1', 'Triplet_Variability_AOD_Empty.2', 'Triplet_Variability_AOD_Empty.3', 'Triplet_Variability_AOD_Empty.4', '340-440_Angstrom_Exponent', '440-675_Angstrom_Exponent[Polar]', 'Solar_Zenith_Angle(Degrees)', 'Exact_Wavelengths_of_AOD(um)_620nm', 'Exact_Wavelengths_of_AOD(um)_555nm', 'Exact_Wavelengths_of_AOD(um)_551nm', 'Exact_Wavelengths_of_AOD(um)_532nm', 'Exact_Wavelengths_of_AOD(um)_531nm', 'Exact_Wavelengths_of_AOD(u

## 4 · Tidy each station and append it to the parquet

The tidying is notebook 01's, as a function applied per station:

* keep AERONET's own column names; only replace characters that break pandas / SQL (`Lunar_Phase_Angle(Degrees)` → `Lunar_Phase_Angle_Degrees`, `Triplet_Variability_Precipitable_Water(cm)` → `Triplet_Variability_Precipitable_Water_cm`, `Measurement_Type(solar or lunar)` → `Measurement_Type_solar_or_lunar`, `440-870_Angstrom_Exponent` → `Angstrom_Exponent_440_870nm`)
* `datetime_utc` is the **measurement time** (not 12:00); `date`, `year`, `month` are its UTC calendar date — a night spans two UTC dates
* drop the placeholder and the all-empty columns found above
* column order: site → time → location → AOD (long → short wavelength) → the rest in AERONET's own order


In [10]:
def clean_col(name: str) -> str:
    name = re.sub(r"(\d{3})-(\d{3})_Angstrom_Exponent", r"Angstrom_Exponent_\1_\2nm", name)
    name = name.replace("[Polar]", "_Polar").replace("(%)", "_percent")
    name = re.sub(r"[\(\[]", "_", name)
    name = re.sub(r"[\)\]]", "", name)
    name = re.sub(r"[^0-9A-Za-z_]", "_", name)
    return re.sub(r"_+", "_", name).strip("_")

In [11]:
def tidy(df: pd.DataFrame) -> pd.DataFrame:
    df = df.drop(columns=empty)
    df["datetime_utc"] = pd.to_datetime(df["Date(dd:mm:yyyy)"] + " " + df["Time(hh:mm:ss)"], format="%d:%m:%Y %H:%M:%S")
    df = df.drop(columns=["Date(dd:mm:yyyy)", "Time(hh:mm:ss)", "site"])     # site: AERONET repeats it in AERONET_Site_Name (checked below)
    df["date"]  = df["datetime_utc"].dt.normalize()
    df["year"]  = df["datetime_utc"].dt.year.astype("int16")
    df["month"] = df["datetime_utc"].dt.month.astype("int8")
    df.columns = [clean_col(c) for c in df.columns]
    for c in ("AERONET_Site_Name", "Data_Quality_Level", "Measurement_Type_solar_or_lunar", "Last_Date_Processed", "PI", "PI_Email"):
        df[c] = df[c].astype("string")
    front = ["AERONET_Site_Name", "datetime_utc", "date", "year", "month", "Day_of_Year", "Day_of_Year_Fraction",
             "Site_Latitude_Degrees", "Site_Longitude_Degrees", "Site_Elevation_m"]
    aod   = sorted([c for c in df.columns if c.startswith("AOD_")], key=lambda c: -int(re.search(r"(\d+)nm", c).group(1)))
    rest  = [c for c in df.columns if c not in front + aod]
    return df[front + aod + rest]

In [12]:
writer, mismatch, written = None, 0, 0
t0 = time.time()
for p in tqdm(station_files, desc="Writing stations"):
    raw = read_station(p)
    mismatch += int((raw["AERONET_Site_Name"] != raw["site"]).sum())
    table = pa.Table.from_pandas(tidy(raw), preserve_index=False)
    if writer is None:                                                 # the first station fixes the schema
        writer = pq.ParquetWriter(OUT_PARQUET, table.schema, compression="zstd")
    writer.write_table(table.cast(writer.schema))                      # one row group per station
    written += table.num_rows
writer.close()

print(f"wrote {OUT_PARQUET}  ({OUT_PARQUET.stat().st_size / 1e6:,.0f} MB, {written:,} rows × {table.num_columns} columns) in {time.time() - t0:,.0f} s")
print(f"site-name mismatches between header and rows: {mismatch}")
print(list(table.column_names))

Writing stations:   0%|          | 0/523 [00:00<?, ?it/s]

Writing stations:   0%|          | 1/523 [00:02<18:46,  2.16s/it]

Writing stations:   1%|          | 3/523 [00:02<05:19,  1.63it/s]

Writing stations:   1%|          | 6/523 [00:02<02:17,  3.75it/s]

Writing stations:   2%|▏         | 8/523 [00:02<01:41,  5.07it/s]

...

Writing stations:  99%|█████████▉| 520/523 [06:49<00:04,  1.46s/it]

Writing stations: 100%|█████████▉| 522/523 [06:50<00:00,  1.08it/s]

Writing stations: 100%|██████████| 523/523 [06:50<00:00,  1.16it/s]

Writing stations: 100%|██████████| 523/523 [06:50<00:00,  1.27it/s]

wrote /home/ubuntu/ai_for_aeronet/DATA/PROCESSED/AERONET_Lunar_AOD_L2_AllPoints_V3.parquet  (1,488 MB, 8,303,752 rows × 78 columns) in 411 s
site-name mismatches between header and rows: 0
['AERONET_Site_Name', 'datetime_utc', 'date', 'year', 'month', 'Day_of_Year', 'Day_of_Year_Fraction', 'Site_Latitude_Degrees', 'Site_Longitude_Degrees', 'Site_Elevation_m', 'AOD_1640nm', 'AOD_1020nm', 'AOD_870nm', 'AOD_865nm', 'AOD_779nm', 'AOD_709nm', 'AOD_681nm', 'AOD_675nm', 'AOD_667nm', 'AOD_560nm', 'AOD_510nm', 'AOD_500nm', 'AOD_490nm', 'AOD_443nm', 'AOD_440nm', 'AOD_412nm', 'Precipitable_Water_cm', 'Triplet_Variability_1640', 'Triplet_Variability_1020', 'Triplet_Variability_870', 'Triplet_Variability_865', 'Triplet_Variability_779', 'Triplet_Variability_675', 'Triplet_Variability_667', 'Triplet_Variability_560', 'Triplet_Variability_510', 'Triplet_Variability_500', 'Triplet_Variability_490', 'Triplet_Variability_443', 'Triplet_Variability_440', 'Triplet_Variability_412', 'Triplet_Variability_Pr

## 5 · Sanity checks

Read back from the parquet, a few columns at a time (the whole table is too big for memory).


In [13]:
aeronet = pd.read_parquet(OUT_PARQUET, columns=["AERONET_Site_Name", "datetime_utc", "AOD_500nm", "Lunar_Phase_Angle_Degrees"])
print(f"sites        : {aeronet['AERONET_Site_Name'].nunique():,}")
print(f"rows         : {len(aeronet):,}  (first pass counted {n_rows:,})")
print(f"time range   : {aeronet['datetime_utc'].min()}  →  {aeronet['datetime_utc'].max()}")
print(f"duplicates   : {aeronet.duplicated(['AERONET_Site_Name', 'datetime_utc']).sum():,}  (site × datetime: AERONET's files repeat some rows verbatim; they are kept as delivered)")
print(f"phase angle  : {aeronet['Lunar_Phase_Angle_Degrees'].min():.1f}° … {aeronet['Lunar_Phase_Angle_Degrees'].max():.1f}°  (negative = waxing, positive = waning, 0 = full moon)")

print("\nnon-null fraction per column (from the first pass):")
kept = counts.drop(index=empty + ["Date(dd:mm:yyyy)", "Time(hh:mm:ss)", "site"])
display((kept / n_rows * 100).round(1).rename("% non-null").to_frame().T.rename(columns=clean_col))

display(pq.ParquetFile(OUT_PARQUET).read_row_group(0).to_pandas().head())        # the first station only

sites        : 523
rows         : 8,303,752  (first pass counted 8,303,752)
time range   : 2014-12-31 09:06:38  →  2026-07-22 05:00:46


duplicates   : 38,471  (site × datetime: AERONET's files repeat some rows verbatim; they are kept as delivered)
phase angle  : -96.6° … 92.3°  (negative = waxing, positive = waning, 0 = full moon)

non-null fraction per column (from the first pass):


,Day_of_Year,Day_of_Year_Fraction,AOD_1640nm,AOD_1020nm,AOD_870nm,AOD_865nm,AOD_779nm,AOD_675nm,AOD_667nm,AOD_560nm,...,Exact_Wavelengths_of_AOD_um_500nm,Exact_Wavelengths_of_AOD_um_490nm,Exact_Wavelengths_of_AOD_um_443nm,Exact_Wavelengths_of_AOD_um_440nm,Exact_Wavelengths_of_AOD_um_412nm,Exact_Wavelengths_of_PW_um_935nm,Exact_Wavelengths_of_AOD_um_681nm,Exact_Wavelengths_of_AOD_um_709nm,PI,PI_Email
% non-null,100.0,100.0,88.2,98.4,95.5,4.3,4.1,95.4,4.3,4.3,...,94.8,4.3,4.2,92.8,4.0,98.7,0.5,0.5,100.0,100.0


,AERONET_Site_Name,datetime_utc,date,year,month,Day_of_Year,Day_of_Year_Fraction,Site_Latitude_Degrees,Site_Longitude_Degrees,Site_Elevation_m,...,Exact_Wavelengths_of_AOD_um_500nm,Exact_Wavelengths_of_AOD_um_490nm,Exact_Wavelengths_of_AOD_um_443nm,Exact_Wavelengths_of_AOD_um_440nm,Exact_Wavelengths_of_AOD_um_412nm,Exact_Wavelengths_of_PW_um_935nm,Exact_Wavelengths_of_AOD_um_681nm,Exact_Wavelengths_of_AOD_um_709nm,PI,PI_Email
0,AAOT,2018-09-21 18:00:46,2018-09-21,2018,9,264,264.750532,45.3139,12.5083,10.0,...,NaN,0.490041,0.442436,NaN,0.412333,0.93683,NaN,NaN,Giuseppe_Zibordi,giuseppe.zibordi@eoscience.eu
1,AAOT,2018-09-21 18:03:46,2018-09-21,2018,9,264,264.752616,45.3139,12.5083,10.0,...,NaN,0.490041,0.442436,NaN,0.412333,0.93683,NaN,NaN,Giuseppe_Zibordi,giuseppe.zibordi@eoscience.eu
2,AAOT,2018-09-21 18:06:46,2018-09-21,2018,9,264,264.754699,45.3139,12.5083,10.0,...,NaN,0.490041,0.442436,NaN,0.412333,0.93683,NaN,NaN,Giuseppe_Zibordi,giuseppe.zibordi@eoscience.eu
3,AAOT,2018-09-21 18:09:46,2018-09-21,2018,9,264,264.756782,45.3139,12.5083,10.0,...,NaN,0.490041,0.442436,NaN,0.412333,0.93683,NaN,NaN,Giuseppe_Zibordi,giuseppe.zibordi@eoscience.eu
4,AAOT,2018-09-21 18:12:46,2018-09-21,2018,9,264,264.758866,45.3139,12.5083,10.0,...,NaN,0.490041,0.442436,NaN,0.412333,0.93683,NaN,NaN,Giuseppe_Zibordi,giuseppe.zibordi@eoscience.eu


### Output schema (data dictionary for the agent's skill / system prompt)

One row per **measurement** (lunar triplet). Column names are AERONET's own with `( ) [ ] -` → `_`.

| column | original AERONET header | unit | meaning |
|---|---|---|---|
| `AERONET_Site_Name` | `AERONET_Site_Name` | – | site name (string) |
| `datetime_utc` | *derived from* `Date(dd:mm:yyyy)` + `Time(hh:mm:ss)` | UTC | time of the measurement |
| `date`, `year`, `month` | *derived* | UTC | calendar date of the measurement (a night spans two UTC dates) |
| `Day_of_Year`, `Day_of_Year_Fraction` | `Day_of_Year`, `Day_of_Year(Fraction)` | day | 1–366, with the time of day as a fraction |
| `Site_Latitude_Degrees`, `Site_Longitude_Degrees`, `Site_Elevation_m` | `Site_Latitude(Degrees)` … | degrees, m | site location |
| `AOD_<λ>nm` | `AOD_<λ>nm` | unitless | night-time aerosol optical depth at λ nm (440, 500, 675, 870, 1020, 1640 and whatever else a site has); NaN = not measured or failed Level 2.0 screening |
| `Precipitable_Water_cm` | `Precipitable_Water(cm)` | cm | column water vapour from the 935 nm channel |
| `Triplet_Variability_<λ>`, `Triplet_Variability_Precipitable_Water_cm` | `Triplet_Variability_<λ>` … | unitless (AOD), cm | max − min of the three AOD values of the triplet: larger at night than by day, largest at large phase angles (faint Moon) |
| `Angstrom_Exponent_<λ1>_<λ2>nm` | `<λ1>-<λ2>_Angstrom_Exponent` | unitless | Ångström exponent between λ1 and λ2 nm |
| `Data_Quality_Level`, `Measurement_Type_solar_or_lunar` | `Data_Quality_Level`, `Measurement_Type(solar or lunar)` | – | always `lev20`, `lunar` |
| `AERONET_Instrument_Number` | `AERONET_Instrument_Number` | – | CIMEL sun/sky/lunar photometer id (Model T) |
| *(dropped)* | `Solar_Zenith_Angle(Degrees)` | degrees | always -999 for lunar rows: dropped as empty |
| `Lunar_Zenith_Angle_Degrees` | `Lunar_Zenith_Angle(Degrees)` | degrees | zenith angle of the Moon |
| `Optical_Air_Mass` | `Optical_Air_Mass` | unitless | air mass of the lunar path (1 = Moon at zenith) |
| `Lunar_Phase_Angle_Degrees` | `Lunar_Phase_Angle(Degrees)` | degrees | 0 = full Moon, −90 / +90 = waxing / waning quarter; measurements exist only within about ±90° |
| `Sensor_Temperature_Degrees_C` | `Sensor_Temperature(Degrees_C)` | °C | detector temperature |
| `Ozone_Dobson`, `NO2_Dobson` | `Ozone(Dobson)`, `NO2(Dobson)` | DU | gas columns used in the processing |
| `Last_Date_Processed`, `Number_of_Wavelengths` | same | – | processing date, number of AOD wavelengths of the instrument |
| `Exact_Wavelengths_of_AOD_um_<λ>nm`, `Exact_Wavelengths_of_PW_um_935nm` | `Exact_Wavelengths_of_AOD(um)_<λ>nm` … | µm | the instrument's actual filter wavelengths |
| `PI`, `PI_Email` | *file header line 5* | – | site principal investigator (AERONET data-use policy: contact / acknowledge) |

Units reference: <https://aeronet.gsfc.nasa.gov/new_web/units.html>
